In [1]:
   import pandas as pd
   from sklearn.linear_model import LinearRegression

   toy = pd.DataFrame({
       "distance_km": [100, 200, 300, 500, 800, 1000],
       "days":        [3,   4,   5,   7,   10,  12],
   })
   
   

In [2]:
model = LinearRegression()
model.fit(toy[["distance_km"]], toy["days"])

model.intercept_, model.coef_

(np.float64(2.000000000000001), array([0.01]))

In [ ]:
model.predict(pd.DataFrame({"distance_km": [650]}))


array([8.5])

In [7]:
   from pathlib import Path

   DATA = Path("../../data/raw/olist")

   orders = pd.read_csv(
       DATA / "olist_orders_dataset.csv",
       parse_dates=["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"],
   )
   items = pd.read_csv(DATA / "olist_order_items_dataset.csv")
   products = pd.read_csv(DATA / "olist_products_dataset.csv")
   customers = pd.read_csv(DATA / "olist_customers_dataset.csv")
   sellers = pd.read_csv(DATA / "olist_sellers_dataset.csv")
   

In [17]:
items.head(1)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29


In [8]:
   delivered = orders[
       (orders["order_status"] == "delivered")
       & orders["order_delivered_customer_date"].notna()
   ]

   items_per_order = items.groupby("order_id").size()
   single_item_orders = items_per_order[items_per_order == 1].index
   one_item = items[items["order_id"].isin(single_item_orders)]

   table = (
       delivered
       .merge(one_item, on="order_id")
       .merge(products, on="product_id")
       .merge(customers, on="customer_id")
       .merge(sellers, on="seller_id")
   )
   len(delivered), len(table)

(96470, 86835)

In [ ]:
   #features
   table["delivery_days"] = (table["order_delivered_customer_date"] - table["order_purchase_timestamp"]).dt.days
   table["estimated_days"] = (table["order_estimated_delivery_date"] - table["order_purchase_timestamp"]).dt.
   table["same_state"] = (table["customer_state"] == table["seller_state"]).astype(int)
   table["late"] = (table["order_delivered_customer_date"] > table["order_estimated_delivery_date"]).astype(int)

In [19]:
   features = ["price", "freight_value", "product_weight_g", "same_state", "estimated_days"]

   data = table[features + ["delivery_days", "late"]]
   before = len(data)
   data = data.dropna()
   before, len(data)

(86835, 86821)

In [22]:
   from sklearn.model_selection import train_test_split

   X = data[features]
   y = data["delivery_days"]

   X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)
   len(X_train), len(X_test)

(69456, 17365)

In [23]:
   from sklearn.tree import DecisionTreeRegressor
   from sklearn.metrics import mean_absolute_error

   tree = DecisionTreeRegressor(random_state=42)
   tree.fit(X_train, y_train)

   mean_absolute_error(y_train, tree.predict(X_train)), mean_absolute_error(y_test, tree.predict(X_test))

(0.418521334673287, 7.271458412017006)

In [24]:
   import numpy as np

   simple_guess = np.full(len(y_test), y_train.mean())
   mean_absolute_error(y_test, simple_guess)

6.570701245989315

In [25]:
mean_absolute_error(y_test, X_test["estimated_days"])


12.794126115750071

In [27]:
linear = LinearRegression()
linear.fit(X_train, y_train)
mean_absolute_error(y_test, linear.predict(X_test))

5.535877733622054

In [28]:
   tree_limited = DecisionTreeRegressor(max_depth=6, random_state=42)
   tree_limited.fit(X_train, y_train)
   mean_absolute_error(y_test, tree_limited.predict(X_test))

5.453921986796242

In [29]:
pd.Series(linear.coef_, index=features)

price              -0.000428
freight_value       0.042919
product_weight_g    0.000052
same_state         -4.109402
estimated_days      0.277092
dtype: float64

In [30]:
y = data["late"]
y.mean()

np.float64(0.08289469137651029)

In [31]:
   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [33]:
from sklearn.metrics import accuracy_score

always_on_time = np.zeros(len(y_test), dtype=int)
accuracy_score(y_test, always_on_time)

0.9171321623956233

In [34]:
   from sklearn.linear_model import LogisticRegression

   clf = LogisticRegression(max_iter=1000)
   clf.fit(X_train, y_train)
   predicted = clf.predict(X_test)
   accuracy_score(y_test, predicted)

0.9170745752951339

In [35]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, predicted)

array([[15925,     1],
       [ 1439,     0]])

In [36]:
from sklearn.metrics import precision_score, recall_score

precision_score(y_test, predicted), recall_score(y_test, predicted)

(0.0, 0.0)

In [37]:
   clf_balanced = LogisticRegression(max_iter=1000, class_weight="balanced")
   clf_balanced.fit(X_train, y_train)
   predicted_balanced = clf_balanced.predict(X_test)

   accuracy_score(y_test, predicted_balanced), precision_score(y_test, predicted_balanced), recall_score(y_test, predicted_balanced)

(0.5808234955369997, 0.11969262828861683, 0.6386379430159833)

In [38]:
   size_columns = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
   sizes = products[size_columns].dropna()
   len(products), len(sizes)

(32951, 32949)

In [39]:
   from sklearn.cluster import KMeans

   kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
   sizes_raw = sizes.copy()
   sizes_raw["group"] = kmeans.fit_predict(sizes)
   sizes_raw.groupby("group")[size_columns].mean().round(0)

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
group,,,,
0,792.0,27.0,14.0,21.0
1,13123.0,51.0,38.0,39.0
2,24705.0,59.0,50.0,47.0
3,5758.0,44.0,26.0,34.0


In [40]:
   from sklearn.preprocessing import StandardScaler

   scaled = StandardScaler().fit_transform(sizes)

   sizes_scaled = sizes.copy()
   sizes_scaled["group"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(scaled)
   sizes_scaled.groupby("group")[size_columns].mean().round(0)

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
group,,,,
0,621.0,22.0,11.0,17.0
1,16392.0,54.0,45.0,43.0
2,2562.0,30.0,32.0,27.0
3,2723.0,52.0,13.0,34.0


In [41]:
   sizes_scaled["group"].value_counts()

group
0    19181
3     6307
2     5523
1     1938
Name: count, dtype: int64